# CS4241 - Introduction to Artificial Intelligence
## Part D: Full RAG Pipeline Implementation (10 Marks)

**Name:** Maureen Amago  
**Index Number:** 10022200180

---
### Pipeline Architecture
```
User Query
    ↓
[STAGE 1] Retrieval       — Search vector store, get top-k chunks
    ↓
[STAGE 2] Context Selection — Rank & filter to fit token budget
    ↓
[STAGE 3] Prompt Builder   — Inject context + strict rules
    ↓
[STAGE 4] LLM              — Ollama (llama3) or smart simulation
    ↓
[STAGE 5] Response         — Display answer + pipeline metrics
```
Each stage is **logged** with timestamps so every decision is traceable.

---
## Cell 1: Imports & Logger

In [ ]:
# Name: Maureen Amago | Index: 10022200180
import re, math, time, logging, os
import numpy as np
import pandas as pd
from datetime import datetime
from pypdf import PdfReader
from collections import Counter

# Load API Key from .env file
if os.path.exists('.env'):
    with open('.env') as f:
        for line in f:
            if line.startswith('GROQ_API_KEY='):
                os.environ['GROQ_API_KEY'] = line.split('=')[1].strip()

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('RAG')
print('✅ Logger ready and API Key loaded.')

---
## Cell 2: Load Document & Create Chunks

In [ ]:
# Name: Maureen Amago | Index: 10022200180
log.info('Loading 2025 Ghana Budget PDF...')
t0 = time.time()

reader = PdfReader('2025-Budget-Statement-and-Economic-Policy_v4.pdf')
raw_text = ''
for i in range(min(50, len(reader.pages))):
    p = reader.pages[i].extract_text()
    if p: raw_text += p + ' '

clean_text = re.sub(r'\s+', ' ', raw_text)
clean_text = re.sub(r'[^\x00-\x7F]+', ' ', clean_text).strip()

# Remove table-of-contents noise lines (lines that are mostly dots)
clean_text = re.sub(r'[.]{4,}', ' ', clean_text)

# Chunk: 500 chars, 50 char overlap
chunks = [clean_text[i:i+500] for i in range(0, len(clean_text), 450)]

log.info(f'Loaded {len(clean_text):,} chars → {len(chunks)} chunks in {time.time()-t0:.2f}s')
print(f'Total chunks : {len(chunks)}')
print(f'First chunk  : {chunks[0][:200]}...')

---
## Cell 3: TF-IDF Vector Store

In [ ]:
# Name: Maureen Amago | Index: 10022200180
log.info('Building TF-IDF vector store...')
t1 = time.time()

STOP = {'the','and','of','in','to','for','is','a','an','on','at','by','as','be','are','this','that','was','with'}

def tokenize(text):
    return [w for w in re.findall(r'\b\w{2,}\b', text.lower()) if w not in STOP]

class VectorStore:
    """Custom TF-IDF vector store with cosine similarity search."""
    def __init__(self, docs):
        self.docs = docs
        all_tok = tokenize(' '.join(docs))
        self.vocab = {w:i for i,(w,_) in enumerate(Counter(all_tok).most_common(5000))}
        self.idf = {w: math.log(len(docs)/(1+sum(1 for d in docs if w in d.lower())))+1
                    for w in self.vocab}
        self.vecs = np.array([self._embed(d) for d in docs])
        log.info(f'Vector store ready | vocab={len(self.vocab):,} | shape={self.vecs.shape}')

    def _embed(self, text):
        v = np.zeros(len(self.vocab))
        toks = tokenize(text)
        if not toks: return v
        for w,c in Counter(toks).items():
            if w in self.vocab:
                v[self.vocab[w]] = (c/len(toks)) * self.idf[w]
        return v

    def search(self, query, k=5):
        qv = self._embed(query)
        sims = []
        for i, cv in enumerate(self.vecs):
            d = np.linalg.norm(qv) * np.linalg.norm(cv)
            sims.append(float(np.dot(qv,cv)/d) if d>0 else 0.0)
        top = np.argsort(sims)[-k:][::-1]
        return [{'doc_id':int(i),'text':self.docs[i],'score':round(sims[i],4)} for i in top]

vs = VectorStore(chunks)
log.info(f'Vector store built in {time.time()-t1:.2f}s')

---
## Cell 4: Full RAG Pipeline

In [ ]:
# Name: Maureen Amago | Index: 10022200180
import requests

PROMPT_TEMPLATE = """You are a Ghana Budget AI specialist for 2025.
Use ONLY the context below to answer the question.
If the answer is NOT in the context, say: "I cannot find that information in the 2025 Budget document."

=== CONTEXT FROM BUDGET DOCUMENT ===
{context}
=====================================

Question: {query}

Answer (2-3 concise sentences based only on the context above):"""


def smart_simulate(context, query):
    """
    When Ollama is not available, produce a grounded answer from the context.
    Finds the most informative sentence in the context that relates to the query.
    """
    query_words = set(tokenize(query))
    # Split context into sentences
    sentences = re.split(r'(?<=[.!?]) +', context)
    # Score each sentence by how many query words it contains
    scored = []
    for s in sentences:
        s_clean = s.strip()
        if len(s_clean) < 30: continue   # skip short fragments
        overlap = len(query_words & set(tokenize(s_clean)))
        scored.append((overlap, s_clean))
    scored.sort(reverse=True)
    
    if not scored or scored[0][0] == 0:
        return "I cannot find that information in the 2025 Budget document."
    
    # Return the top 2 most relevant sentences
    best = [s for _, s in scored[:2]]
    return "Based on the 2025 Budget document: " + " ".join(best)


def call_llm(prompt, context, query):
    """Try Groq API (Llama3). Fall back to smart simulation."""
    api_key = os.environ.get('GROQ_API_KEY')
    if not api_key:
        return smart_simulate(context, query), 'smart-simulation (no key)'
    
    try:
        # Groq OpenAI-compatible endpoint
        url = "https://api.groq.com/openai/v1/chat/completions"
        headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        }
        data = {
            "model": "llama-3.3-70b-versatile",
            "messages": [
                {"role": "system", "content": "You are a Ghana Budget AI specialist."},
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.1
        }
        r = requests.post(url, headers=headers, json=data, timeout=15)
        if r.status_code == 200:
            return r.json()['choices'][0]['message']['content'].strip(), 'groq:llama-3.3-70b'
        else:
            log.warning(f"Groq API Error: {r.status_code} - {r.text}")
    except Exception as e:
        log.error(f"LLM Call failed: {e}")
        
    return smart_simulate(context, query), 'smart-simulation'


def rag_pipeline(user_query, k=5, top_k_context=2):
    pipeline_start = time.time()
    SEP = '='*65
    print(SEP)
    print(f'  RAG PIPELINE  |  {datetime.now().strftime("%H:%M:%S")}')
    print(SEP)
    print(f'  QUERY: {user_query}')
    print(SEP)

    # ── STAGE 1: RETRIEVAL ━━━━━━━━━━━━━━━━━━
    log.info('STAGE 1 ▶ Retrieval started')
    t = time.time()
    results = vs.search(user_query, k=k)
    log.info(f'STAGE 1 ✓ Retrieved {k} docs in {time.time()-t:.3f}s')

    print(f'\n📂 STAGE 1 — RETRIEVED DOCUMENTS (Top {k})')
    print('-'*65)
    display(pd.DataFrame([{
        'Rank': i+1,
        'Doc ID': r['doc_id'],
        'Similarity Score': r['score'],
        'Preview': r['text'][:90].replace('\n',' ') + '...'
    } for i,r in enumerate(results)]))

    # ── STAGE 2: CONTEXT SELECTION ━━━━━━━━━━━━━━━
    log.info(f'STAGE 2 ▶ Selecting top {top_k_context} of {k} chunks')
    selected = results[:top_k_context]
    context  = '\n---\n'.join([r['text'] for r in selected])
    log.info(f'STAGE 2 ✓ Context length: {len(context)} chars')

    print(f'\n✂️  STAGE 2 — CONTEXT SELECTION (keeping top {top_k_context})')
    print('-'*65)
    for r in selected:
        print(f'  [Doc {r["doc_id"]}] Score={r["score"]}\n  {r["text"]}\n')

    # ── STAGE 3: PROMPT BUILDING ━━━━━━━━━━━━━━━━━
    log.info('STAGE 3 ▶ Building prompt')
    prompt = PROMPT_TEMPLATE.format(context=context, query=user_query)
    log.info(f'STAGE 3 ✓ Prompt length: {len(prompt)} chars')

    print(f'\n📝 STAGE 3 — FINAL PROMPT SENT TO LLM')
    print('-'*65)
    print(prompt)

    # ── STAGE 4: LLM ━━━━━━━━━━━━━━━━━━━━━
    log.info('STAGE 4 ▶ Calling LLM...')
    t = time.time()
    answer, model = call_llm(prompt, context, user_query)
    log.info(f'STAGE 4 ✓ Response from [{model}] in {time.time()-t:.2f}s')

    # ── STAGE 5: RESPONSE ━━━━━━━━━━━━━━━━━━━━
    total = time.time() - pipeline_start
    print(f'\n💬 STAGE 5 — FINAL RESPONSE')
    print('-'*65)
    print(answer)
    print(f'\n⏱️  Done in {total:.2f}s  |  Model: {model}')
    print(SEP + '\n')
    return answer


---
## Cell 5: Interactive — Ask Your Own Question

In [ ]:
# Name: Maureen Amago | Index: 10022200180
user_question = input('Ask the Ghana Budget AI: ')
rag_pipeline(user_question)

---
## Cell 6: Pre-set Test Queries (for Examiner)

In [ ]:
# Name: Maureen Amago | Index: 10022200180
test_queries = [
    'What is the projected economic growth for Ghana in 2025?',
    'What are the key revenue measures in the 2025 budget?',
    'What is the budget for the Ghana space program?'   # Hallucination test
]
for q in test_queries:
    rag_pipeline(q)